In [21]:
from pathlib import Path
import tkinter as tk
from tkinter import filedialog
import pandas as pd


root = tk.Tk()
# root.withdraw()
filename = filedialog.askopenfilename(
    title="Select production failures Parquet file",
    filetypes=[("Parquet files", "*.parquet *.pq"), ("All files", "*.*")],
)
root.destroy()

if not filename:
    raise FileNotFoundError("No production failures file was selected.")

dataset_path = Path(filename)
if dataset_path.suffix.lower() not in {".parquet", ".pq"}:
    raise ValueError("The selected file must be a Parquet file.")

production_failures = pd.read_parquet(dataset_path)
production_failures.head(10)

,uwi,formatted_uwi,well_name,field_operating_area,failure_start_date,failure_end_date,cause,cause_detail,cause_category,action_taken,failure_detail,failure_description,failure_system,failure_system_category,failure_comments,failure_type_category,cause_comments,failure_type
0,19G092704225W300,19G/09-27-042-25W3/00,NBRI WINTER HZ 19C10-27-15A9-27-42-25,Winter,2026-10-26,None,Fatigue,NaN,None,NaN,Rods Routine - Upset,NaN,NaN,None,NaN,None,NaN,Rods
1,100121303701W400,100/12-13-037-01W4/00,CALTEX_RES 100 MH PROVOST 12-13-37-1,Bodo East,2026-10-26,None,Fatigue,NaN,None,NaN,Rods Routine - Broken Corod body,NaN,NaN,None,NaN,None,NaN,Rods
2,192162703528W300,192/16-27-035-28W3/00,NBRI CACTUS LAKE V2U DD 6C9-27-1B16-27-35-28,Cactus Lake,2026-10-26,None,Corrosion,Rods,None,NaN,Rods Routine - Broken Corod body,NaN,NaN,None,Broken corod @ 268m,None,NaN,Rods
3,193082103528W300,193/08-21-035-28W3/00,NBRI CACTUS LAKE V2U DD 11A9-21-4D8-21-35-28,Cactus Lake,2026-10-26,None,Corrosion,Rods,None,NaN,Rods Routine - Broken Corod body,NaN,NaN,None,Broken corod @ 258m,None,NaN,Rods
4,104082403729W300,104/08-24-037-29W3/00,CALTEX 104 HZ COSINE LAKE 8-24-37-29,Cosine,2026-10-26,None,Fatigue,NaN,None,NaN,Bent/Broken Rotor,NaN,NaN,None,Broken rotor 3' from top,None,NaN,Pump
5,102141703528W300,102/14-17-035-28W3/00,SRL CACTUS LAKE DD 3C14-17-35-28W3 (28B14-17),Cactus Lake,2026-10-26,None,NaN,NaN,None,NaN,Hole,NaN,NaN,None,Hole in the upset on joint 88 rite above the s...,None,NaN,Tubing
6,102122103701W400,102/12-21-037-01W4/00,CALTEX_RES 102 HZ PROVOST 12-21-37-1,Bodo East,2026-10-25,None,Corrosion,Rods,None,NaN,Rod - Broken Corod Body,NaN,NaN,None,NaN,None,NaN,Rods
7,101011203628W300,101/01-12-036-28W3/00,SRL CACTUS LAKE DD 2C1-12-36-28 (3C1-12),Cactus Lake,2026-10-25,None,Fatigue,NaN,None,NaN,Rods Routine - Polished Rod,NaN,NaN,None,NaN,None,NaN,Rods
8,19E092704225W300,19E/09-27-042-25W3/00,NBRI WINTER HZ 11C10-27-14D9-27-42-25,Winter,2026-10-25,None,Fatigue,NaN,None,NaN,Rods Routine - Polished Rod,NaN,NaN,None,NaN,None,NaN,Rods
9,191142503328W300,191/14-25-033-28W3/00,NBRI ET AL COURT V1U DD 6B3-36-3C14-25-33-28,Court,2026-10-25,None,Fatigue,NaN,None,NaN,Parted,NaN,NaN,None,Top of joint 55 broken threads,None,NaN,Tubing


In [23]:
production_failures.drop(columns=[
    "formatted_uwi", "well_name", "cause_category", "action_taken", "failure_description", "failure_system", "failure_comments", "failure_system_category", "failure_type_category", "cause_comments"
    ], inplace=True)


C:\Users\tranm\AppData\Local\Temp\ipykernel_18152\188505777.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  production_failures.drop(columns=[


In [24]:
production_failures.columns

Index(['uwi', 'field_operating_area', 'failure_start_date', 'failure_end_date',
       'cause', 'cause_detail', 'failure_detail', 'failure_type'],
      dtype='str')

In [25]:
production_failures_for_encoding = production_failures.drop(columns=[
    "uwi", "field_operating_area", "failure_start_date", "failure_end_date", "failure_detail"
    ])
production_failures_for_encoding.columns

Index(['cause', 'cause_detail', 'failure_type'], dtype='str')

In [26]:
production_failures_encoded = production_failures_for_encoding.apply(lambda col: pd.factorize(col))
production_failures_encoded_df = pd.DataFrame({col: encoded[0] for col, encoded in production_failures_encoded.items()})
production_failures_encoded_df.head()

,cause,cause_detail,failure_type
0,0,-1,0
1,0,-1,0
2,1,0,0
3,1,0,0
4,0,-1,1


In [27]:
# combine the encoded production failures with the original dataframe
production_failures_combined = pd.concat([production_failures, production_failures_encoded_df], axis=1)
production_failures_combined.head()

,uwi,field_operating_area,failure_start_date,failure_end_date,cause,cause_detail,failure_detail,failure_type,cause,cause_detail,failure_type
0,19G092704225W300,Winter,2026-10-26,None,Fatigue,NaN,Rods Routine - Upset,Rods,0,-1,0
1,100121303701W400,Bodo East,2026-10-26,None,Fatigue,NaN,Rods Routine - Broken Corod body,Rods,0,-1,0
2,192162703528W300,Cactus Lake,2026-10-26,None,Corrosion,Rods,Rods Routine - Broken Corod body,Rods,1,0,0
3,193082103528W300,Cactus Lake,2026-10-26,None,Corrosion,Rods,Rods Routine - Broken Corod body,Rods,1,0,0
4,104082403729W300,Cosine,2026-10-26,None,Fatigue,NaN,Bent/Broken Rotor,Pump,0,-1,1
